# Day 072 — Exercise 3: transcribe_audio

**What you'll build:** `transcribe_audio(source, transcribe_fn=None, model='base') -> dict` — the core transcription function with `transcribe_fn=None` mock injection.

**Why it matters:** This is the hub of the AudioTranscriber pipeline. All class methods call it. Getting the injection pattern right makes every downstream method automatically testable.

In [ ]:
import os, tempfile
_MOCK_RESULT = {
    'text': ' Hello world. This is a test of speech recognition.',
    'language': 'en',
    'segments': [
        {'id': 0, 'start': 0.0, 'end': 3.2, 'text': ' Hello world.',
         'avg_logprob': -0.25, 'no_speech_prob': 0.01},
        {'id': 1, 'start': 3.2, 'end': 7.8,
         'text': ' This is a test of speech recognition.',
         'avg_logprob': -0.30, 'no_speech_prob': 0.02},
    ],
}
_mock_transcribe = lambda source: _MOCK_RESULT


## Task

Implement `transcribe_audio`:

1. If `transcribe_fn is not None`: `return transcribe_fn(source)`
2. `import whisper as _whisper; mdl = _whisper.load_model(model)`
3. If `isinstance(source, (bytes, bytearray))`: write to `NamedTemporaryFile(suffix='.wav', delete=False)`, call `mdl.transcribe(tmp)` in a try/finally that unlinks the temp file
4. Else: `return mdl.transcribe(str(source))`

The checks all use `transcribe_fn`, so the whisper path is not tested here.

## Your Implementation

In [ ]:
def transcribe_audio(source, transcribe_fn=None, model: str = 'base') -> dict:
    """Transcribe an audio source using openai-whisper.

    Args:
        source:        File path (str/Path), bytes, or numpy array
        transcribe_fn: callable(source) -> dict for testing
        model:         Whisper model size: tiny, base, small, medium, large
    Returns:
        dict with keys: text (str), language (str), segments (list)
    """
    raise NotImplementedError


In [ ]:
def transcribe_audio(source, transcribe_fn=None, model='base'):
    if transcribe_fn is not None:
        return transcribe_fn(source)
    import whisper as _whisper
    mdl = _whisper.load_model(model)
    if isinstance(source, (bytes, bytearray)):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            f.write(source)
            tmp = f.name
        try:
            return mdl.transcribe(tmp)
        finally:
            os.unlink(tmp)
    return mdl.transcribe(str(source))


## Automated checks

In [ ]:

score, total = 0, 5
try:
    # returns dict with correct keys
    result = transcribe_audio(b'fake audio', transcribe_fn=_mock_transcribe)
    assert isinstance(result, dict)
    assert 'text' in result and 'language' in result and 'segments' in result
    score += 1; print("✅ returns dict with text/language/segments")

    # text is non-empty
    assert isinstance(result['text'], str) and len(result['text']) > 0
    score += 1; print("✅ text is non-empty string")

    # language is a string
    assert isinstance(result['language'], str)
    score += 1; print("✅ language is a string")

    # transcribe_fn called with the source
    captured = {}
    def _cap(src):
        captured['src'] = src
        return _MOCK_RESULT
    transcribe_audio(b'test bytes', transcribe_fn=_cap)
    assert captured.get('src') == b'test bytes'
    score += 1; print("✅ transcribe_fn receives the source argument")

    # path source also works via mock
    result2 = transcribe_audio('fake/path.mp3', transcribe_fn=_mock_transcribe)
    assert result2['language'] == 'en'
    score += 1; print("✅ string path source forwarded correctly")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def transcribe_audio(source, transcribe_fn=None, model='base'):
    if transcribe_fn is not None:
        return transcribe_fn(source)
    import whisper as _whisper
    mdl = _whisper.load_model(model)
    if isinstance(source, (bytes, bytearray)):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            f.write(source)
            tmp = f.name
        try:
            return mdl.transcribe(tmp)
        finally:
            os.unlink(tmp)
    return mdl.transcribe(str(source))
```

**Why `try/finally` for the temp file?** If `mdl.transcribe` raises an exception (bad audio, model error), the temp file would be left on disk without the `finally`. Always clean up temporary files regardless of whether the operation succeeds.

</details>